In [3]:
import sklearn
import numpy as np
import pandas
import torch
import sklearn.datasets
import sklearn.preprocessing
import helpers
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import OneClassSVM
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score, precision_recall_curve, roc_auc_score, f1_score, make_scorer, auc, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Credit Card Dataset:
Class 0 : Normal  
Class 1 : Fraud

Context: Credit card must be able to recognize fraudulent credit card transactions so that customers are not charged for items that they did not purchase.
- Label of transactions are given, classes: {Normal, Fraud}  ==> looking at **supervised anomaly detection algorithm**
- Performance Metric: as requested, **Area under Precision-Recall curve (AOPRC)**
- Given a new transaction X, model should output M(X) = 0 if X is deemed normal, and 1 if its fraudulent


In [10]:
X_credit_card, Y, classes = helpers.load_creditcard_dataset()
Y.value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

Insights: Severe class imbalance:
  - Solutions: SMOTE sampling, balance weight classifieres that set weights proportional to class frequency (OC-SVM, Autoencoders, Isolation Forest)

Split data into train, validate, and test set

In [11]:
normal_transactions = X_credit_card[X_credit_card["Class"]==0]
X_normal = normal_transactions.drop("Class", axis=1)
Y_normal =  normal_transactions["Class"].copy()

fraud_transactions = X_credit_card[X_credit_card["Class"] == 1]
X_fraud = fraud_transactions.drop("Class", axis=1)
Y_fraud = fraud_transactions["Class"].copy()

X_train, X_test, Y_train, Y_test = train_test_split(X_normal, Y_normal, test_size=0.2, random_state=40)
X_test = pandas.concat([X_test, X_fraud[:250]], ignore_index=True)
Y_test_with_anomalies = pandas.concat([Y_test, Y_fraud[:250]], ignore_index=True)

X_train, X_validate, Y_train, Y_validate = train_test_split(X_train, Y_train, test_size=0.2, random_state=40)
X_validate_with_anomalies = pandas.concat(
    [X_validate, X_fraud[250:]], ignore_index=True)
Y_validate_with_anomalies = pandas.concat(
    [Y_validate, Y_fraud[250:]], ignore_index=True)


# Data Analyzing:


- All continuous values
Check value distribution of each columns (see where regularization may be needed)

In [6]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 181961 entries, 92733 to 264531
Data columns (total 30 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    181961 non-null  float64
 1   V1      181961 non-null  float64
 2   V2      181961 non-null  float64
 3   V3      181961 non-null  float64
 4   V4      181961 non-null  float64
 5   V5      181961 non-null  float64
 6   V6      181961 non-null  float64
 7   V7      181961 non-null  float64
 8   V8      181961 non-null  float64
 9   V9      181961 non-null  float64
 10  V10     181961 non-null  float64
 11  V11     181961 non-null  float64
 12  V12     181961 non-null  float64
 13  V13     181961 non-null  float64
 14  V14     181961 non-null  float64
 15  V15     181961 non-null  float64
 16  V16     181961 non-null  float64
 17  V17     181961 non-null  float64
 18  V18     181961 non-null  float64
 19  V19     181961 non-null  float64
 20  V20     181961 non-null  float64
 21  V21     181

# OC_SVM from Sklearn: Anomaly Detection

In [25]:
def output_formatter(predictions):
    predictions = np.where(predictions == 1, 0, 1)
    return predictions

In [31]:
oc_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", OneClassSVM(kernel="rbf", gamma=0.009, nu=0.001))
])
oc_svm_clf.fit(X_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svm_clf', OneClassSVM(gamma=0.009, nu=0.001))])

As gamma increases: less precision, more recall ==> slower convergence
As gamma decreases: more precision, less recall ==> faster convergence

as nu increases: less precision, more recall
as nu decreases: more precision, less recall


In [32]:
prediction_with_anomalies = output_formatter(oc_svm_clf.predict(X_validate_with_anomalies))
prediction_with_anomalies

array([0, 0, 0, ..., 0, 0, 0], shape=(45733,))

In [283]:
Y_validate_with_anomalies

0        0
1        0
2        0
3        0
4        0
        ..
45728    1
45729    1
45730    1
45731    1
45732    1
Name: Class, Length: 45733, dtype: int64

Confusion Matrix for OCSVM

| TN,      FP  |      
| FN,      TP  |     

In [33]:
confusion_matrix(Y_validate_with_anomalies, prediction_with_anomalies)

array([[45366,   125],
       [   84,   158]])

In [34]:
precision_score(Y_validate_with_anomalies, prediction_with_anomalies)

0.558303886925795

In [35]:
recall_score(Y_validate_with_anomalies, prediction_with_anomalies)

0.6528925619834711

In [36]:
f1_score(Y_validate_with_anomalies, prediction_with_anomalies)

0.6019047619047619

OneClassSVM(kernel="rbf", gamma=0.009, nu=0.001)    
F1 = .60

**Testing with current best OC-SVM model params**

In [37]:
test_prediction = output_formatter(oc_svm_clf.predict(X_test))

In [38]:
confusion_matrix(Y_test_with_anomalies, test_prediction)

array([[56699,   164],
       [   91,   159]])

In [39]:
precision_score(Y_test_with_anomalies, test_prediction)

0.49226006191950467

In [40]:
recall_score(Y_test_with_anomalies, test_prediction)

0.636

In [41]:
f1_score(Y_test_with_anomalies, test_prediction)

0.5549738219895288

In [1]:
oc_svm_clf.get_params()

NameError: name 'oc_svm_clf' is not defined

In [13]:
False in (X_normal.index == Y_normal.index)

False

# Isolation Forest: Anomaly detection with Scikit Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest

# Sanity check for correct partition
all([all(X_credit_card.index == Y.index), all(X_normal.index == Y_normal.index), all(X_fraud.index == Y_fraud.index)])

True

Keep a separate subsets for testing purposes

In [56]:
X_train, X_test, Y_train, Y_test = train_test_split(X_normal, Y_normal, test_size=0.2, random_state=42)
X_test_with_anomalies = pandas.concat([X_test, X_fraud], ignore_index=True)
Y_test_with_anomalies = pandas.concat([Y_test, Y_fraud], ignore_index=True)

Different parameters to tune Isolation Model

In [57]:
contanmination_factors = [float(len(Y_fraud) / len(Y_normal)), 0.01, 0.001]
num_itrees = [25, 50, 100]
max_samples = [128, 'auto', 512]
max_features = [4, 6, 8]

In [58]:
from itertools import product
models = []
avg_scores = []
random_states = [40,50,60]
for (contanmination_factor, n_tree, max_sample, max_feature) in product(contanmination_factors, num_itrees, max_samples, max_features):
    model = IsolationForest(n_estimators=n_tree, contamination=contanmination_factor, max_samples=max_sample, max_features=max_feature, n_jobs=-1, random_state=42)
    p, r, f1 = 0,0,0
    models.append(model)
    for rs in random_states:
        X_trainset, X_validate, Y_trainset, Y_validate = train_test_split(X_train, Y_train, test_size=0.2, random_state=rs)
        X_validate_with_anomalies = pandas.concat(
            [X_trainset, X_fraud], ignore_index=True)
        Y_validate_with_anomalies = pandas.concat(
            [Y_trainset, Y_fraud], ignore_index=True)
        
        model.fit(X_trainset)
        predictions = output_formatter(
            model.predict(X_validate_with_anomalies))
        
        p += precision_score(Y_validate_with_anomalies, predictions)
        r += recall_score(Y_validate_with_anomalies, predictions)
        f1 += f1_score(Y_validate_with_anomalies, predictions)
    
    avg_scores.append([p/len(random_states), r/len(random_states), f1/len(random_states)])
    

In [69]:
avg_scores = np.array(avg_scores)
avg_scores

array([[0.34614029, 0.34349593, 0.3443976 ],
       [0.40791637, 0.44715447, 0.42611302],
       [0.25686683, 0.22831978, 0.24140871],
       [0.41875825, 0.46341463, 0.43975506],
       [0.2845304 , 0.25474255, 0.26880442],
       [0.31535575, 0.29945799, 0.30688947],
       [0.43866408, 0.50067751, 0.46758795],
       [0.37349154, 0.3800813 , 0.37675151],
       [0.34284831, 0.33536585, 0.33894643],
       [0.32931718, 0.31436314, 0.32157276],
       [0.36109636, 0.36314363, 0.36200351],
       [0.28244344, 0.25948509, 0.27005589],
       [0.37608077, 0.38821138, 0.38185343],
       [0.29423982, 0.26761518, 0.28024599],
       [0.31509985, 0.3001355 , 0.30705904],
       [0.36157436, 0.36382114, 0.36259105],
       [0.34519748, 0.33807588, 0.34155098],
       [0.3286394 , 0.31436314, 0.32126723],
       [0.3887308 , 0.40718157, 0.39774   ],
       [0.35231882, 0.34891599, 0.3505504 ],
       [0.35984793, 0.3604336 , 0.36009132],
       [0.40945507, 0.44715447, 0.42716676],
       [0.

3 best performing models

In [70]:
indices = np.argsort(avg_scores[:, 2])[-3:][::-1]
indices

array([ 6,  3, 60])

In [71]:
best_models = [models[i] for i in indices]
best_models

[IsolationForest(contamination=0.0017304750013189597, max_features=4,
                 max_samples=512, n_estimators=25, n_jobs=-1, random_state=42),
 IsolationForest(contamination=0.0017304750013189597, max_features=4,
                 n_estimators=25, n_jobs=-1, random_state=42),
 IsolationForest(contamination=0.001, max_features=4, max_samples=512,
                 n_estimators=25, n_jobs=-1, random_state=42)]

In [72]:
best_if_model = best_models[0]

In [75]:
pred = output_formatter(best_if_model.predict(X_test_with_anomalies))
[precision_score(Y_test_with_anomalies, pred), recall_score(
    Y_test_with_anomalies, pred), f1_score(Y_test_with_anomalies, pred)]

[0.6965317919075145, 0.4898373983739837, 0.5751789976133651]

In [76]:
scores = []
for rs in range(1,100):
    X_trainset, X_test, Y_trainset, Y_test = train_test_split(
        X_normal, Y_normal, test_size=0.2, random_state=rs)
    X_test_with_anomalies = pandas.concat(
        [X_test, X_fraud], ignore_index=True)
    Y_test_with_anomalies = pandas.concat(
        [Y_test, Y_fraud], ignore_index=True)
    best_if_model.fit(X_trainset)
    predictions = output_formatter(
        best_if_model.predict(X_test_with_anomalies))

    scores.append([precision_score(Y_test_with_anomalies, predictions), recall_score(Y_test_with_anomalies, predictions), f1_score(Y_test_with_anomalies, predictions)])

In [78]:
scores = np.array(scores)
np.mean(scores, axis=0)

array([0.70709137, 0.48439681, 0.57374692])

Attempt to check whether feature scaling affect training results:

In [80]:
scores_feature_scaled = []
iso_forest = Pipeline([
    ("scaler", StandardScaler()),
    ("if_clf", best_if_model)
])
for rs in range(1, 100):
    X_trainset, X_test, Y_trainset, Y_test = train_test_split(
        X_normal, Y_normal, test_size=0.2, random_state=rs)
    X_test_with_anomalies = pandas.concat(
        [X_test, X_fraud], ignore_index=True)
    Y_test_with_anomalies = pandas.concat(
        [Y_test, Y_fraud], ignore_index=True)
    iso_forest.fit(X_trainset)
    predictions = output_formatter(
        iso_forest.predict(X_test_with_anomalies))

    scores_feature_scaled.append([precision_score(Y_test_with_anomalies, predictions), recall_score(
        Y_test_with_anomalies, predictions), f1_score(Y_test_with_anomalies, predictions)])

In [81]:
scores_feature_scaled

[[0.6786885245901639, 0.42073170731707316, 0.5194479297365119],
 [0.6933333333333334, 0.5284552845528455, 0.5997693194925029],
 [0.7220708446866485, 0.5386178861788617, 0.6169965075669382],
 [0.7203166226912929, 0.5548780487804879, 0.6268656716417911],
 [0.6789297658862876, 0.41260162601626016, 0.5132743362831859],
 [0.7205479452054795, 0.5345528455284553, 0.6137689614935823],
 [0.6485623003194888, 0.41260162601626016, 0.5043478260869565],
 [0.7084639498432602, 0.45934959349593496, 0.5573366214549939],
 [0.7097701149425287, 0.5020325203252033, 0.5880952380952381],
 [0.7092391304347826, 0.5304878048780488, 0.6069767441860465],
 [0.6497890295358649, 0.3130081300813008, 0.4224965706447188],
 [0.7368421052631579, 0.5691056910569106, 0.6422018348623854],
 [0.5771812080536913, 0.34959349593495936, 0.43544303797468353],
 [0.7013422818791947, 0.4247967479674797, 0.529113924050633],
 [0.7068965517241379, 0.4166666666666667, 0.5242966751918159],
 [0.7073863636363636, 0.5060975609756098, 0.590047

In [82]:
scores_feature_scaled = np.array(scores_feature_scaled)
np.mean(scores_feature_scaled, axis=0)

array([0.70709137, 0.48439681, 0.57374692])

In [83]:
best_if_model.get_params()

{'bootstrap': False,
 'contamination': 0.0017304750013189597,
 'max_features': 4,
 'max_samples': 512,
 'n_estimators': 25,
 'n_jobs': -1,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [84]:
confusion_matrix(Y_test_with_anomalies, predictions)

array([[56758,   105],
       [  251,   241]])

Conclusion: No significant changes to performance when scaling features

**Should modify to give more weights to recall score, as depending on domain, might be more costly to let anomaly go undetected**

Current model performances:  
    - OC-SVM( kernel="rbf", gamma=0.009, nu=0.001 ): F1_score ~ 0.5549  
    - Isolation Forest ('contamination': 0.0017304750013189597, 'max_features': 4, 'max_samples': 512, 'n_estimators': 25, 'n_jobs': -1, ) : F1_score ~ 0.57374692

# Autoencoder: Anomaly Detection